In [2]:
#Loading Daata
import sys

sys.path.append("..")

from src.data import load_raw, imposed_split
from sklearn.metrics import f1_score, accuracy_score
import emoji
import pandas as pd

full, test = load_raw()
train, val = imposed_split(full)



In [3]:
# dump the tweets

for t in train.tweet.sample(15, random_state=0): print(repr(t))
print("-" * 60)
for t in test.tweet.sample(15, random_state=0): print(repr(t))

'"#MannequinChallenge انا خلوا حياتي نفسه تثبت على مود محدد عشان انا اثبت"'
'عطر غوتشي بامبو" الجديد يحاكي الطيور https://t.co/F9PRduoecU https://t.co/eGRH39QeLq"'
'"#ستبقى_حلب#ستنتصر_حلب(فدعا ربه أني مغلوب فأنتصر)والله لتنصرن ولو بعد #حين"'
'"@DROS_ASEER 📚شرح أصول السُنّة للإمام أحمد✒الشيخ سلطان العرابي📍جامع الراجحي بأبها🕗عصر الجمعة ١٨-٢-١٤٣٨🕗وبعد الم… https://t.co/DqFHkhsqQ9"'
'مااحلى شعارك ومدافعك ياقنرررزز #ارسنال'
'"RT @BassamJaara: الطائرات الروسية تحرق الاخضر واليابس والجنود الروس يسرحون ويمرحون  في شوارع حلب المحتلة ومدن اخرى لأنهم يشعرون بالامان!!"'
'إقالة لبيب السباعى من الطب الشرعى وتعيين د إحسان كميل جورجي بدلاً منه" الى مزبلة التاريخ وشكرا'
'@aldanahk @girlposts دام انه مال هاري بوتر بستعمله 💅🏼'
'"بطلبٍ من #هشام_حداد.. #رامي_عياش يحقّق حلم لين حايك https://t.co/xH63acQjcr @HichamHaddad57 @RamyAyach"'
'"@shihanh010 بوش والا اوباما😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂"'
'"في المستقبل الحريري يبحث التطور

In [4]:
import re

markers = {
    "quotes":    lambda s: s.str.startswith('"'),
    "RT":        lambda s: s.str.contains("RT @"),
    "url":       lambda s: s.str.contains("http"),
    "mention":   lambda s: s.str.contains("@"),
    "hashtag":   lambda s: s.str.contains("#"),
    "emoji":     lambda s: s.map(emoji.emoji_count) > 0,
    "laugh":     lambda s: s.str.contains("ههه"),
    "!!":        lambda s: s.str.contains(r"[!؟?]{2,}"),
    "elongated": lambda s: s.map(lambda t: bool(re.search(r"(\w)\1\1", t))),
    "latin":     lambda s: s.str.contains("[a-zA-Z]"),
}
rows = []
for name, f in markers.items():
    has = f(train.tweet)
    v = f(val.tweet)
    rows.append({"marker": name,
                 "% train": has.mean(), "% test": f(test.tweet).mean(),
                 "sarc if present": train.y[has].mean(),
                 "sarc if absent": train.y[~has].mean(),
                 "1-rule F1 val": max(f1_score(val.y, v), f1_score(val.y, ~v))})
pd.DataFrame(rows).round(3)

,marker,% train,% test,sarc if present,sarc if absent,1-rule F1 val
0,quotes,0.608,0.008,0.123,0.250,0.301
1,RT,0.111,0.000,0.161,0.174,0.283
2,url,0.384,0.147,0.066,0.239,0.360
3,mention,0.234,0.008,0.166,0.175,0.266
4,hashtag,0.476,0.330,0.148,0.195,0.298
5,emoji,0.120,0.163,0.165,0.174,0.285
6,laugh,0.012,0.032,0.606,0.168,0.288
7,!!,0.043,0.090,0.360,0.164,0.279
8,elongated,0.060,0.132,0.292,0.165,0.278
9,latin,0.554,0.239,0.109,0.252,0.327


In [ ]:
# dedup train * test
print("exact duplicates:", full.tweet.duplicated().sum())
print("train ∩ test:", len(set(full.tweet) & set(test.tweet)))
c = full.groupby("tweet").y.nunique()

print("same tweet, different labels:", (c > 1).sum())

full[full.tweet.isin(c[c > 1].index)].to_csv(
    "../results/contradictory_duplicates.csv", index=False)

exact duplicates: 61
train ∩ test: 2
same tweet, different labels: 17


In [6]:
for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, len(d), round(d.y.mean(), 3))
pd.concat({n: d.groupby("dialect").y.agg(["mean", "size"])
           for n, d in [("train", train), ("val", val), ("test", test)]}, axis=1).round(3)

train 11293 0.173
val 1255 0.173
test 3000 0.274


train          val        test      
          mean  size   mean size   mean  size
dialect                                      
egypt    0.346  2415  0.362  260  0.507   306
gulf     0.248   576  0.206   68  0.177   322
levant   0.214   569  0.291   55  0.298    47
magreb   0.395    38  0.000    5  0.000     2
msa      0.109  7695  0.107  867  0.256  2323

In [7]:
pd.concat({n: d.groupby("sentiment").y.agg(["mean", "size"])
           for n, d in [("train", train), ("test", test)]}, axis=1).round(3)

train         test      
            mean  size   mean  size
sentiment                          
NEG        0.420  4156  0.423  1677
NEU        0.030  5163  0.142   748
POS        0.027  1974  0.010   575

In [9]:
# never sarcastic
z = [0] * len(val)
print("accuracy:", accuracy_score(val.y, z), "| F1:", f1_score(val.y, z))

# => accuracy rewards ignoring sarcasm, so we use the F1 of the sarcastic class from now on.

accuracy: 0.8270916334661355 | F1: 0.0
